# 04: LangGraph, Stateful Agents & Workflows

**12 components, one section each**

This notebook follows the [LangGraph API map](assets/langgraph_api_landscape.png) component by component. Each section covers one component and runs its most useful methods as examples. Anything else in that component is listed at the end of its section so you know it exists.

| # | Component | Examples we run |
|---|---|---|
| 1 | Graph builder | `StateGraph(State)`, `.add_node()`, `.add_edge()`, `.add_conditional_edges()`, `.compile()` |
| 2 | State & reducers | `TypedDict`, `add_messages`, `MessagesState` |
| 3 | Control flow | `Command(goto=, update=)`, `Send(node, arg)`, `interrupt(value)` |
| 4 | Run a compiled graph | `.invoke()`, `.stream()` |
| 5 | Streaming | `'updates'`, `'messages'` |
| 6 | Persistence | `InMemorySaver`, `thread_id` |
| 7 | Long-term memory | `InMemoryStore`, `.put(ns, key, val)` |
| 8 | Functional API | `@entrypoint`, `@task` |
| 9 | Runtime & config | `Runtime[Context]` |
| 10 | Prebuilt | `ToolNode`, `tools_condition` |
| 11 | Composition | subgraph as node |
| 12 | Deploy & remote | `langgraph dev` |

The mental model is three things: **State** (shared memory), **Nodes** (functions that read state and return updates), **Edges** (who runs next). Everything else is a variation.

## Setup (run once)

The LangSmith key is **optional** here, so press Enter to skip it. If you set it, every run in this notebook is traced for free.

Everything runs on `gpt-4o-mini`, so it is cheap (a full run of this notebook costs a few cents). Swap `MODEL` for any provider you like.

In [ ]:
%pip install -q -U langgraph langchain langchain-openai pydantic

In [12]:
import os, getpass

def set_env(var: str):
    """Ask for a secret only if it is not already in the environment."""
    if not os.environ.get(var):
        val = getpass.getpass(f"{var} (Enter to skip): ")
        if val:
            os.environ[var] = val

set_env("OPENAI_API_KEY")
set_env("LANGSMITH_API_KEY")

if os.environ.get("LANGSMITH_API_KEY"):
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ["LANGSMITH_PROJECT"] = "meetup-langgraph"
    print("LangSmith tracing ON -> project: meetup-langgraph")
else:
    print("LangSmith tracing OFF")

MODEL = "openai:gpt-4o-mini"   # provider:model - change freely

LangSmith tracing ON -> project: meetup-langgraph


In [13]:
from langchain.chat_models import init_chat_model
llm = init_chat_model(MODEL, temperature=0)

## 1. Graph builder
You describe a graph with a builder, then turn it into something runnable:

- `StateGraph(State)` is the builder, given a state schema
- `.add_node(name, fn)` adds a node: a function `state -> partial state update`
- `.add_edge(a, b)` runs `b` after `a`. `START` and `END` are special
- `.add_conditional_edges(src, router)` lets a router function pick the next node
- `.compile()` turns the builder into a runnable graph

We start without an LLM on purpose: it shows that LangGraph is just orchestration. Nodes can be *anything*.

**Also worth knowing:** `.add_sequence()`, `START`, `END`, `context_schema=`, `input_schema` / `output_schema`

In [14]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    text: str

def to_upper(state: State) -> dict:
    return {"text": state["text"].upper()}

def add_excitement(state: State) -> dict:
    return {"text": state["text"] + "!!!"}

builder = StateGraph(State)
builder.add_node("to_upper", to_upper)
builder.add_node("add_excitement", add_excitement)
builder.add_edge(START, "to_upper")
builder.add_edge("to_upper", "add_excitement")
builder.add_edge("add_excitement", END)

graph = builder.compile()
print(graph.invoke({"text": "hello meetup"}))

{'text': 'HELLO MEETUP!!!'}


### Conditional edges: your code picks the path
A **router function** looks at the state and returns the *name of the next node*. Here, if the question contains a digit we go to `math_node`, otherwise to `chat_node`. This is a workflow, because **your code** decides the path, not the model.

```
START ──route()──► math_node ──► END
              └──► chat_node ──► END
```

The text printed at the end is a diagram you can paste into https://mermaid.live.

In [15]:
from typing import Literal

class RouterState(TypedDict):
    question: str
    answer: str

# Two simple nodes: each one writes an answer
def math_node(state: RouterState) -> dict:
    return {"answer": "This looks like a math question."}

def chat_node(state: RouterState) -> dict:
    return {"answer": "This looks like normal chat."}

# The router: read the state, return the NAME of the next node
def route(state: RouterState) -> Literal["math_node", "chat_node"]:
    has_digit = any(ch.isdigit() for ch in state["question"])
    return "math_node" if has_digit else "chat_node"

rb = StateGraph(RouterState)
rb.add_node("math_node", math_node)
rb.add_node("chat_node", chat_node)
rb.add_conditional_edges(START, route)      # after START, ask route() where to go
rb.add_edge("math_node", END)
rb.add_edge("chat_node", END)
router_graph = rb.compile()

print(router_graph.get_graph().draw_mermaid())
print(router_graph.invoke({"question": "What is 48 divided by 6?"})["answer"])
print(router_graph.invoke({"question": "I had a rough day at work."})["answer"])

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	math_node(math_node)
	chat_node(chat_node)
	__end__([<p>__end__</p>]):::last
	__start__ -.-> chat_node;
	__start__ -.-> math_node;
	chat_node --> __end__;
	math_node --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

This looks like a math question.
This looks like normal chat.


## 2. State & reducers
State is the memory shared by all nodes. By default a node's return value **overwrites** a key. A **reducer** says how to **merge** instead.

- `TypedDict` is the simplest way to declare state (we used it in section 1)
- `MessagesState` is a built-in state with one key, `messages`
- `add_messages` is the reducer behind it: returning `{"messages": [new]}` *appends* rather than overwrites

**Also worth knowing:** Pydantic `BaseModel`, `@dataclass`, `Annotated[list, operator.add]`, `Overwrite`, `RemoveMessage`, `REMOVE_ALL_MESSAGES`

In [ ]:
from langgraph.graph import MessagesState

def chatbot(state: MessagesState) -> dict:
    return {"messages": [llm.invoke(state["messages"])]}



b = StateGraph(MessagesState)
b.add_node("chatbot", chatbot)
b.add_edge(START, "chatbot")
b.add_edge("chatbot", END)
chat_graph = b.compile()

out = chat_graph.invoke({"messages": [("user", "Hi! I'm Priya. Give me a fun fact about graphs.")]})
for m in out["messages"]:
    m.pretty_print()

print("\n--- A second invoke knows nothing about the first (no memory yet; see section 6) ---")
out2 = chat_graph.invoke({"messages": [("user", "What is my name?")]})
print(out2["messages"][-1].content)

================================ Human Message =================================

Hi! I'm Priya. Give me a fun fact about graphs.
================================== Ai Message ==================================

Hi Priya! Here's a fun fact about graphs: The famous "Seven Bridges of Königsberg" problem, posed by mathematician Leonhard Euler in 1736, is considered one of the first problems in graph theory. The challenge was to find a walk through the city of Königsberg that would cross each of its seven bridges exactly once. Euler proved that it was impossible, laying the groundwork for the field of topology and graph theory!

--- A second invoke knows nothing about the first (no memory yet; see section 6) ---
I'm sorry, but I don't have access to personal information about you unless you've shared it in this conversation. How can I assist you today?


### `add_messages` on its own
Here is the reducer without a graph. It merges two lists of messages, and `MessagesState` is just a `TypedDict` that uses it.

In [19]:
from typing import Annotated
from langchain_core.messages import HumanMessage, AIMessage
from langgraph.graph.message import add_messages

merged = add_messages([HumanMessage("Hi")], [AIMessage("Hello! How can I help?")])
print([m.type for m in merged])                 # ['human', 'ai'] : appended, not overwritten

class ChatState(TypedDict):
    messages: Annotated[list, add_messages]     # this is all MessagesState is

['human', 'ai']


## 3. Control flow
Three tools decide what happens next:

- `Command(goto=, update=)`: a node returns *where to go next* and *what to save* in one value
- `Send(node, arg)`: start one parallel run of a node per item, when the number of tasks is only known at runtime
- `interrupt(value)`: pause the graph and wait for a human

**Also worth knowing:** `Command(resume=)`, `Command.PARENT`, `recursion_limit`, `RetryPolicy`, `CachePolicy`, `defer=True`

### `Command`: a supervisor that routes
A **supervisor** node decides which worker runs next, and every worker hands control back to it. Each node returns `Command(goto=..., update=...)`. Here the supervisor follows simple rules (you could swap them for an LLM). Always set a `recursion_limit` as a safety net.

```
START → supervisor → researcher → supervisor → writer → supervisor → END
```

In [20]:
from langgraph.types import Command

class TeamState(TypedDict):
    topic: str
    notes: str
    article: str

def supervisor(state: TeamState) -> Command[Literal["researcher", "writer", "__end__"]]:
    if not state.get("notes"):            # nothing researched yet
        return Command(goto="researcher")
    if not state.get("article"):          # research done, no article yet
        return Command(goto="writer")
    return Command(goto=END)              # everything is done

def researcher(state: TeamState) -> Command[Literal["supervisor"]]:
    notes = llm.invoke(f"Give 3 short facts about: {state['topic']}").content
    return Command(goto="supervisor", update={"notes": notes})       # save the notes, go back to the supervisor

def writer(state: TeamState) -> Command[Literal["supervisor"]]:
    article = llm.invoke(f"Write a 3-sentence article using these notes:\n{state['notes']}").content
    return Command(goto="supervisor", update={"article": article})

sb = StateGraph(TeamState)
sb.add_node("supervisor", supervisor)
sb.add_node("researcher", researcher)
sb.add_node("writer", writer)
sb.add_edge(START, "supervisor")          # the only edge we add: Command decides the rest
team = sb.compile()

result = team.invoke({"topic": "why graphs suit agent workflows"}, config={"recursion_limit": 12})
print(result["article"])

Graphs are particularly effective at illustrating dynamic relationships and interactions among entities, making them essential for agent workflows that navigate complex data networks like social networks or supply chains. With the ability to perform flexible and efficient querying, graph databases empower agents to swiftly access pertinent information, thereby enhancing decision-making and responsiveness. Furthermore, the inherent scalability and adaptability of graph structures enable these workflows to evolve seamlessly as new data and relationships emerge, ensuring sustained relevance and operational efficiency.


### `Send`: one task per item
Sometimes you only know at runtime *how many* parallel tasks you need (here: one joke per subject). A conditional edge can return a list of `Send(node, input)`, which starts one run of that node for each item. A **reducer** (`operator.add`) collects all the results into one list.

```
START ──► make_joke("cats")    ┐
      ──► make_joke("coffee")  ├──► jokes = [..., ..., ...] ──► END
      ──► make_joke("Mondays") ┘
```

In [21]:
import operator
from langgraph.types import Send

class JokesState(TypedDict):
    subjects: list[str]
    jokes: Annotated[list[str], operator.add]     # reducer: every result is added to this list

class OneSubject(TypedDict):
    subject: str                                   # what each parallel run receives

def fan_out(state: JokesState):
    return [Send("make_joke", {"subject": s}) for s in state["subjects"]]   # one Send per subject

def make_joke(state: OneSubject) -> dict:
    joke = llm.invoke(f"One short joke about {state['subject']}.").content
    return {"jokes": [joke]}

mb = StateGraph(JokesState)
mb.add_node("make_joke", make_joke)
mb.add_conditional_edges(START, fan_out, ["make_joke"])
mb.add_edge("make_joke", END)
mapreduce_graph = mb.compile()

out = mapreduce_graph.invoke({"subjects": ["cats", "coffee", "Mondays"]})
for joke in out["jokes"]:
    print("-", joke)

- Why was the cat sitting on the computer? 

Because it wanted to keep an eye on the mouse!
- Why do coffee beans never get in trouble? Because they know how to espresso themselves!
- Why did Monday apply for a job? 

Because it wanted to be a little less "week" and a little more "end"!


### `interrupt`: pause for a human
`interrupt(value)` **pauses** the graph and hands `value` to your app (for example, to show in a UI). The state is saved in a checkpointer (section 6 explains it; interrupts need one), so the human can answer seconds or days later. You resume with `Command(resume=answer)`, and that answer becomes the return value of `interrupt()`.

This is how you let an agent *propose* a $120 refund but never *execute* it without a human click.

```
START → approval_node ── pauses at interrupt() ── human says "approve" ──► END
```

In [22]:
from langgraph.types import interrupt
from langgraph.checkpoint.memory import InMemorySaver

class RefundState(TypedDict):
    request: str
    result: str

def approval_node(state: RefundState) -> dict:
    answer = interrupt(f"Approve this refund? {state['request']}")     # the graph pauses here
    if answer == "approve":                                           # the human's reply arrives here
        return {"result": "Refund issued."}
    return {"result": "Refund rejected."}

hb = StateGraph(RefundState)
hb.add_node("approval", approval_node)
hb.add_edge(START, "approval")
hb.add_edge("approval", END)
hitl_graph = hb.compile(checkpointer=InMemorySaver())      # a checkpointer is required for interrupts

cfg = {"configurable": {"thread_id": "refund-1"}}
paused = hitl_graph.invoke({"request": "Refund $120 for order #8841 (damaged item)"}, cfg)
print("PAUSED. Shown to the human:", paused["__interrupt__"][0].value)
print("Waiting at node:", hitl_graph.get_state(cfg).next)

PAUSED. Shown to the human: Approve this refund? Refund $120 for order #8841 (damaged item)
Waiting at node: ('approval',)


In [23]:
# ... the human clicks "Approve" (could be a different process, hours later) ...
final = hitl_graph.invoke(Command(resume="approve"), cfg)
print(final["result"])

# Try it: use a fresh thread_id and resume with "reject" instead.

Refund issued.


## 4. Run a compiled graph
Once compiled, a graph is a Runnable with the same methods as everything else in LangChain:

- `.invoke()` runs to the end and returns the final state
- `.stream()` yields progress while the graph runs

**Also worth knowing:** `.astream()`, `.batch()`, `.get_state_history()`, `.update_state()`, `.get_graph().draw_mermaid()`, `.with_config()`

In [27]:
run_graph = builder.compile(checkpointer=InMemorySaver())     # `builder` is the to_upper -> add_excitement graph from section 1
cfg = {"configurable": {"thread_id": "run-demo"}}

print("invoke:", run_graph.invoke({"text": "hello meetup"}, cfg))

print("\nstream:")
for state in run_graph.stream({"text": "streaming demo"}, cfg):
    print(" ", state)



invoke: {'text': 'HELLO MEETUP!!!'}

stream:
  {'to_upper': {'text': 'STREAMING DEMO'}}
  {'add_excitement': {'text': 'STREAMING DEMO!!!'}}


## 5. Streaming
Watching progress matters for UX. Pass `stream_mode=` to `.stream()`. The two modes we use:

- `"updates"`: only what each node changed
- `"messages"`: LLM tokens as they are generated

**Also worth knowing:** `'values'` (full state after each step), `'custom'`, `'debug'`, `'checkpoints'`, `'tasks'`, `get_stream_writer()`, `subgraphs=True`

In [29]:
print("=== stream_mode='updates' ===")
for event in graph.stream({"text": "streaming demo"}, stream_mode="updates"):
    print(event)

print("\n=== stream_mode='messages' (token streaming) ===")
for chunk, meta in chat_graph.stream(
    {"messages": [("user", "Write one sentence about the moon.")]}, stream_mode="messages"
):
    if meta["langgraph_node"] == "chatbot":
        print(chunk.content, end="", flush=True)
print()

=== stream_mode='updates' ===
{'to_upper': {'text': 'STREAMING DEMO'}}
{'add_excitement': {'text': 'STREAMING DEMO!!!'}}

=== stream_mode='messages' (token streaming) ===
The moon, Earth's only natural satellite, captivates observers with its silvery glow and influences tides through its gravitational pull.


## 6. Persistence
A **checkpointer** saves a snapshot of the state after every step.

- `InMemorySaver` stores the snapshots (use `SqliteSaver` / `PostgresSaver` in real apps)
- `thread_id` names a conversation. Same thread means same memory; a different thread starts fresh

That gives you memory, resumability, human approval (section 3) and time travel.

**Also worth knowing:** `SqliteSaver`, `PostgresSaver`, `checkpoint_id`, `durability=`, `StateSnapshot`

In [30]:
memory_graph = b.compile(checkpointer=InMemorySaver())      # same chatbot builder `b` as section 2

t1 = {"configurable": {"thread_id": "thread-1"}}
t2 = {"configurable": {"thread_id": "thread-2"}}

memory_graph.invoke({"messages": [("user", "Hi, I'm Priya.")]}, t1)
print("thread-1:", memory_graph.invoke({"messages": [("user", "What's my name?")]}, t1)["messages"][-1].content)
print("thread-2:", memory_graph.invoke({"messages": [("user", "What's my name?")]}, t2)["messages"][-1].content)

snapshot = memory_graph.get_state(t1)
print("\nStored messages in thread-1:", len(snapshot.values["messages"]))

thread-1: Your name is Priya. How can I help you today?
thread-2: I'm sorry, but I don't have access to personal information about you unless you've shared it in this conversation. How can I assist you today?

Stored messages in thread-1: 4


## 7. Long-term memory
- A **checkpointer** is memory *within* a thread (short-term).
- A **store** is memory *across* threads, organised by namespace (for example, one per user). It suits preferences and facts.

`InMemoryStore` is the store, and `.put(namespace, key, value)` saves an item.

**Also worth knowing:** `PostgresStore`, `.get()`, `.search(query=)`, `index={embed, dims}`, `get_store()`, `InjectedStore`

In [33]:
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()
store.put(("memories", "priya"), "diet", {"fact": "I'm vegetarian."})   # (namespace, key, value)

print(store.get(("memories", "priya"), "diet").value)
print([item.value for item in store.search(("memories", "priya"))])

{'fact': "I'm vegetarian."}
[{'fact': "I'm vegetarian."}]


In [34]:
import uuid
from langchain_core.runnables import RunnableConfig
from langgraph.store.base import BaseStore

def chat(state: MessagesState, config: RunnableConfig, *, store: BaseStore) -> dict:
    namespace = ("memories", config["configurable"]["user_id"])

    # 1. read what we already remember about this user
    remembered = [item.value["text"] for item in store.search(namespace)]

    # 2. answer, using those memories
    reply = llm.invoke([("system", f"Things the user told you earlier: {remembered}")] + state["messages"])

    # 3. remember this message for next time
    store.put(namespace, str(uuid.uuid4()), {"text": state["messages"][-1].content})
    return {"messages": [reply]}

store = InMemoryStore()
mem = StateGraph(MessagesState)
mem.add_node("chat", chat)
mem.add_edge(START, "chat")
mem.add_edge("chat", END)
mem_graph = mem.compile(checkpointer=InMemorySaver(), store=store)

thread_a = {"configurable": {"thread_id": "A", "user_id": "priya"}}
thread_b = {"configurable": {"thread_id": "B", "user_id": "priya"}}   # a NEW thread, but the SAME user

mem_graph.invoke({"messages": [("user", "I'm vegetarian and I love trekking.")]}, thread_a)
answer = mem_graph.invoke({"messages": [("user", "Suggest a dinner and a weekend plan for me.")]}, thread_b)
print(answer["messages"][-1].content)

print("\nStored memories:", [item.value["text"] for item in store.search(("memories", "priya"))])

For dinner, how about a delicious vegetarian stir-fry? You can use a mix of your favorite vegetables like bell peppers, broccoli, and snap peas, sautéed with tofu in a savory soy sauce and garlic mixture. Serve it over brown rice or quinoa for a wholesome meal. You could also add some cashews or sesame seeds for extra crunch!

As for your weekend plan, consider a trekking adventure! Here’s a suggestion:

**Saturday:**
- **Morning:** Start your day with a hearty breakfast, perhaps oatmeal topped with fruits and nuts. Then head out for a morning trek at a nearby trail. Choose a route that offers beautiful scenery and perhaps a waterfall or viewpoint.
- **Afternoon:** Pack a vegetarian picnic lunch to enjoy at a scenic spot during your hike. Bring along some sandwiches with hummus and veggies, fresh fruit, and some trail mix.
- **Evening:** After your trek, unwind with a relaxing evening at home. You could prepare a new vegetarian recipe or order from a local vegetarian restaurant. 

**Su

## 8. Functional API
The same engine, written as ordinary Python instead of a graph:

- `@entrypoint` marks the workflow function
- `@task` marks a unit of work. Its result is saved, so a resumed run does not repeat it. Calling a task returns a future, and `.result()` waits for it.

Use it when your flow is easy to write as plain `if` / `for` code and you still want persistence and human approval.

**Also worth knowing:** `future.result()`, `entrypoint.final`, `previous`, `checkpointer=`, `store=`

In [35]:
from langgraph.func import entrypoint, task

@task
def write_joke(topic: str) -> str:
    return llm.invoke(f"Tell a very short joke about {topic}.").content

@task
def rate_joke(joke: str) -> str:
    return llm.invoke(f"Rate this joke from 1 to 10 in one short sentence:\n{joke}").content

@entrypoint(checkpointer=InMemorySaver())
def joke_workflow(topic: str) -> dict:
    joke = write_joke(topic).result()       # a task returns a future; .result() waits for it
    rating = rate_joke(joke).result()
    return {"joke": joke, "rating": rating}

cfg = {"configurable": {"thread_id": "joke-1"}}
print(joke_workflow.invoke("cats", cfg))

{'joke': 'Why was the cat sitting on the computer? \n\nBecause it wanted to keep an eye on the mouse!', 'rating': "I'd rate it a 7 for its clever wordplay and relatable humor!"}


## 9. Runtime & config
Some data belongs to a *run*, not to the conversation: a user name, a model choice, a database handle. Declare it with `context_schema=`, pass it as `context=` when you invoke, and read it inside a node with `Runtime[Context]`. It stays out of the state, so it is never saved in checkpoints.

**Also worth knowing:** `runtime.context`, `runtime.store`, `get_runtime()`, `get_config()`, `RunnableConfig`, `configurable`

In [36]:
from dataclasses import dataclass
from langgraph.runtime import Runtime

@dataclass
class Context:
    user_name: str

def greet(state: MessagesState, runtime: Runtime[Context]) -> dict:
    system = ("system", f"Address the user as {runtime.context.user_name}. Be brief.")
    return {"messages": [llm.invoke([system] + state["messages"])]}

cb = StateGraph(MessagesState, context_schema=Context)
cb.add_node("greet", greet)
cb.add_edge(START, "greet")
cb.add_edge("greet", END)
context_graph = cb.compile()

out = context_graph.invoke({"messages": [("user", "Say hello.")]}, context=Context(user_name="Priya"))
print(out["messages"][-1].content)

Hello, Priya!


## 10. Prebuilt
Two ready-made pieces build the classic tool loop (ReAct):

- `ToolNode` runs the tool calls found in the last AI message
- `tools_condition` is a router: go to `"tools"` if the model asked for tools, otherwise `END`

Remember the manual loop from the LangChain notebook? Here it is as a **graph with a cycle**: `agent → (tool calls?) → tools → agent → … → END`. This picture is every agent you will ever see.

LangChain's `create_agent` returns the **same kind of compiled graph**, plus extras (structured responses, middleware, ...). Start with it, and hand-build with `StateGraph` when you need custom routing, parallel steps or multiple agents.

**Also worth knowing:** `InjectedState`, `InjectedStore`, `ToolRuntime`, `ValidationNode`, `create_agent` (from `langchain.agents`)

In [37]:
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode, tools_condition

@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    return f"It is 31°C and sunny in {city}."

@tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers."""
    return a * b

tools = [get_weather, multiply]
llm_with_tools = llm.bind_tools(tools)

def agent_node(state: MessagesState) -> dict:
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

ab = StateGraph(MessagesState)
ab.add_node("agent", agent_node)
ab.add_node("tools", ToolNode(tools))
ab.add_edge(START, "agent")
ab.add_conditional_edges("agent", tools_condition)   # -> "tools" if tool calls, else END
ab.add_edge("tools", "agent")                         # the loop
react_graph = ab.compile()

print(react_graph.get_graph().draw_mermaid())
res = react_graph.invoke({"messages": [("user", "What is 12.5 * 8 and what's the weather in Pune?")]})
for m in res["messages"]:
    m.pretty_print()

# create_agent builds the same agent <-> tools loop for you
from langchain.agents import create_agent
prebuilt = create_agent(model=MODEL, tools=tools, system_prompt="Be concise.")
print(prebuilt.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	agent(agent)
	tools(tools)
	__end__([<p>__end__</p>]):::last
	__start__ --> agent;
	agent -.-> __end__;
	agent -.-> tools;
	tools --> agent;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

================================ Human Message =================================

What is 12.5 * 8 and what's the weather in Pune?
================================== Ai Message ==================================
Tool Calls:
  multiply (call_9xRzj7cP6UP7gyBCbTV4Yv7y)
 Call ID: call_9xRzj7cP6UP7gyBCbTV4Yv7y
  Args:
    a: 12.5
    b: 8
  get_weather (call_OVUUhWMBk3zzT8bsM5ppBemh)
 Call ID: call_OVUUhWMBk3zzT8bsM5ppBemh
  Args:
    city: Pune
================================= Tool Message =================================
Name: multiply

100.0
================================= Tool Message =================================
Name: get_weather

It is 3

## 11. Composition
Big systems are built by combining small graphs. The pattern we use is **subgraph as node**: a compiled graph can be added as a node of another graph, so a reusable piece of logic is written once and shared.

**Also worth knowing:** `Command.PARENT` handoff, supervisor pattern (section 3), swarm / handoffs, map-reduce via `Send` (section 3), parallel fan-out (two edges leaving the same node)

In [ ]:
class CleanState(TypedDict):
    text: str

sub = StateGraph(CleanState)
sub.add_node("strip", lambda s: {"text": " ".join(s["text"].split())})
sub.add_node("lower", lambda s: {"text": s["text"].lower()})
sub.add_edge(START, "strip")
sub.add_edge("strip", "lower")
sub.add_edge("lower", END)
cleaner = sub.compile()

parent = StateGraph(CleanState)
parent.add_node("cleaner", cleaner)                                  # <- subgraph as a node
parent.add_node("shout", lambda s: {"text": s["text"].upper() + "!"})
parent.add_edge(START, "cleaner")
parent.add_edge("cleaner", "shout")
parent.add_edge("shout", END)

print(parent.compile().invoke({"text": "   LangGraph   ROCKS   "}))

## 12. Deploy & remote
This section is about the terminal, not the notebook. `langgraph dev` starts a local LangGraph server for your graph and opens **Studio**, a UI to visualise, step through and fork runs. It reads a `langgraph.json` file that points at your graph.

Put the graph from section 10 in a file called `my_agent.py`, then add this `langgraph.json` next to it:

```json
{
  "dependencies": ["."],
  "graphs": { "react_agent": "./my_agent.py:react_graph" },
  "env": ".env"
}
```

and run `langgraph dev`.

| Need | LangGraph answer |
|---|---|
| Durable state | `PostgresSaver` / `SqliteSaver` checkpointers |
| Run as an API | LangGraph server (`langgraph dev` locally, `langgraph up` in Docker) |
| Call it from other code | `get_client()`, `client.threads`, `client.runs.stream`, or `RemoteGraph` |
| Observability | Set the LangSmith env vars: every node, tool and LLM call is traced automatically |

**Also worth knowing:** `langgraph.json`, `langgraph up`, `get_client()`, `client.threads`, `client.runs.stream`, `RemoteGraph`, Studio

---
## Recap

1. **State + nodes + edges** is the whole model (`StateGraph`, `.add_node()`, `.add_edge()`, `.compile()`); **reducers** like `add_messages` control how updates merge.
2. **Conditional edges and cycles** give you routing and agent loops; `ToolNode` + `tools_condition` is the classic one.
3. **`Command`** = update + route in one return, **`Send`** = dynamic parallelism, **`interrupt`** = pause for a human.
4. **Checkpointers** (`InMemorySaver` + `thread_id`) give you memory and resumability; a **store** adds memory across threads.
5. The **Functional API** (`@entrypoint`, `@task`) is the same engine written as plain Python.
6. **Runtime context** carries per-run data outside the state; **subgraphs** compose graphs.
7. `langgraph dev` takes the same graph to a local server and Studio.

### How the three tools fit together
> Build components with **LangChain** → wire them into reliable, stateful flows with **LangGraph** → watch, test and improve everything with **LangSmith**.

### Optional exercises
- Add a third answer path to the router in section 1 (for example, a `greeting_node` when the question contains "hello").
- Add a `reviewer` worker to the supervisor team in section 3 that runs after the writer.
- Resume the refund graph with `"reject"` on a fresh `thread_id` and read the result.
- Turn on LangSmith tracing and open the supervisor run: spot every `Command` hop in the trace.